In [23]:
import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

package = joblib.load('asse_artifacts/student_model_package.pkl')

# 2. استخراج المكونات من داخل القاموس
model = package['model']    # الموديل الحقيقي اللي فيه predict_proba
scaler = package['scaler']       # الميزان اللي بيحول البيانات
feature_columns = package['columns'] # ترتيب الأعمدة الصح
explainer = joblib.load("asse_artifacts/my_lr_explainer.pkl")
X_test_scaled = joblib.load("asse_artifacts/X_test_scaled.pkl")
preprocessor  = joblib.load('asse_artifacts/data_preprocessor.pkl')


print("All Done")

All Done


In [11]:
X_test = pd.read_csv("test_processed.csv")

In [13]:
# ──  Resource catalog (20 items) ──────────────────────────────────────────
resources = [
    # Study & Academic
    {'id': 'R01', 'title': 'Khan Academy — Mathematics Fundamentals',
     'topic': 'math', 'difficulty': 'beginner', 'learning_style': 'visual',
     'url': 'https://www.khanacademy.org/math',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 1},

    {'id': 'R02', 'title': 'Coursera — Learning How to Learn',
     'topic': 'study_skills', 'difficulty': 'beginner', 'learning_style': 'video',
     'url': 'https://www.coursera.org/learn/learning-how-to-learn',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 1},

    {'id': 'R03', 'title': 'Pomodoro Technique — Study Timer App',
     'topic': 'productivity', 'difficulty': 'beginner', 'learning_style': 'practical',
     'url': 'https://pomofocus.io',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 0},

    {'id': 'R04', 'title': 'Anki — Spaced Repetition Flashcards',
     'topic': 'memory', 'difficulty': 'beginner', 'learning_style': 'active_recall',
     'url': 'https://apps.ankiweb.net',
     'motivation': 0, 'hours_studied': 1, 'sleep': 0, 'resources_access': 0},

    {'id': 'R05', 'title': 'MIT OpenCourseWare — Science & Engineering',
     'topic': 'science', 'difficulty': 'advanced', 'learning_style': 'reading',
     'url': 'https://ocw.mit.edu',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 1},

    {'id': 'R06', 'title': 'Crash Course — Science & Humanities Videos',
     'topic': 'general', 'difficulty': 'intermediate', 'learning_style': 'visual',
     'url': 'https://www.youtube.com/crashcourse',
     'motivation': 1, 'hours_studied': 0, 'sleep': 0, 'resources_access': 0},

    {'id': 'R07', 'title': 'Quizlet — Interactive Study Sets',
     'topic': 'study_skills', 'difficulty': 'beginner', 'learning_style': 'active_recall',
     'url': 'https://quizlet.com',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 0},

    # Wellness & Sleep
    {'id': 'R08', 'title': 'Sleep Foundation — Healthy Sleep Guide for Students',
     'topic': 'wellness', 'difficulty': 'beginner', 'learning_style': 'reading',
     'url': 'https://www.sleepfoundation.org/teens-and-sleep',
     'motivation': 0, 'hours_studied': 0, 'sleep': 1, 'resources_access': 0},

    {'id': 'R09', 'title': 'Headspace — Student Mindfulness & Stress Relief',
     'topic': 'wellness', 'difficulty': 'beginner', 'learning_style': 'audio',
     'url': 'https://www.headspace.com/students',
     'motivation': 0, 'hours_studied': 0, 'sleep': 1, 'resources_access': 0},

    {'id': 'R10', 'title': 'Calm — Sleep & Relaxation for Students',
     'topic': 'wellness', 'difficulty': 'beginner', 'learning_style': 'audio',
     'url': 'https://www.calm.com',
     'motivation': 0, 'hours_studied': 0, 'sleep': 1, 'resources_access': 0},

    # Motivation & Engagement
    {'id': 'R11', 'title': 'TED-Ed — Motivational Student Talks',
     'topic': 'motivation', 'difficulty': 'beginner', 'learning_style': 'visual',
     'url': 'https://ed.ted.com',
     'motivation': 1, 'hours_studied': 0, 'sleep': 0, 'resources_access': 0},

    {'id': 'R12', 'title': 'Growth Mindset — Carol Dweck (Book Summary)',
     'topic': 'motivation', 'difficulty': 'intermediate', 'learning_style': 'reading',
     'url': 'https://www.mindsetonline.com',
     'motivation': 1, 'hours_studied': 0, 'sleep': 0, 'resources_access': 0},

    {'id': 'R13', 'title': 'Goal Setting Worksheet — SMART Goals for Students',
     'topic': 'motivation', 'difficulty': 'beginner', 'learning_style': 'practical',
     'url': 'https://www.smartgoalsguide.com',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 0},

    # Low-resource access
    {'id': 'R14', 'title': 'Project Gutenberg — Free Books & Study Materials',
     'topic': 'general', 'difficulty': 'intermediate', 'learning_style': 'reading',
     'url': 'https://www.gutenberg.org',
     'motivation': 0, 'hours_studied': 1, 'sleep': 0, 'resources_access': 1},

    {'id': 'R15', 'title': 'YouTube EDU — Free Educational Video Library',
     'topic': 'general', 'difficulty': 'beginner', 'learning_style': 'visual',
     'url': 'https://www.youtube.com/education',
     'motivation': 0, 'hours_studied': 0, 'sleep': 0, 'resources_access': 1},

    {'id': 'R16', 'title': 'OpenStax — Free Peer-Reviewed Textbooks',
     'topic': 'science', 'difficulty': 'advanced', 'learning_style': 'reading',
     'url': 'https://openstax.org',
     'motivation': 0, 'hours_studied': 1, 'sleep': 0, 'resources_access': 1},

    # Advanced / High motivation
    {'id': 'R17', 'title': 'edX — University-Level Online Courses',
     'topic': 'general', 'difficulty': 'advanced', 'learning_style': 'video',
     'url': 'https://www.edx.org',
     'motivation': 1, 'hours_studied': 1, 'sleep': 1, 'resources_access': 1},

    {'id': 'R18', 'title': 'Brilliant.org — Problem-Solving & Critical Thinking',
     'topic': 'math', 'difficulty': 'advanced', 'learning_style': 'active_recall',
     'url': 'https://brilliant.org',
     'motivation': 1, 'hours_studied': 1, 'sleep': 1, 'resources_access': 1},

    {'id': 'R19', 'title': 'Duolingo — Language Learning (Cognitive Boost)',
     'topic': 'cognitive', 'difficulty': 'beginner', 'learning_style': 'gamified',
     'url': 'https://www.duolingo.com',
     'motivation': 1, 'hours_studied': 0, 'sleep': 0, 'resources_access': 0},

    {'id': 'R20', 'title': 'Notion — Student Study Planner Template',
     'topic': 'productivity', 'difficulty': 'beginner', 'learning_style': 'practical',
     'url': 'https://www.notion.so/templates/student-planner',
     'motivation': 1, 'hours_studied': 1, 'sleep': 0, 'resources_access': 0},
]

with open('asse_artifacts/resources_catalog.json', 'w') as f:
    json.dump(resources, f, indent=2)

print(f'Resource catalog: {len(resources)} items saved.')
print('Saved: asse_artifacts/resources_catalog.json')

Resource catalog: 20 items saved.
Saved: asse_artifacts/resources_catalog.json


In [7]:
def build_smart_student_vector(student_raw, shap_values, feature_names):
    """
    بناء ناقل (Vector) للطالب بناءً على بياناته + رؤية الـ SHAP
    """
    # تحويل قيم SHAP لـ Series
    shap_series = pd.Series(shap_values, index=feature_names)
    
    # تحديد الاحتياجات بناءً على القيم السالبة في SHAP (الأسهم الزرقاء)
    needs_motivation = 1 if (student_raw.get('Motivation_Level') == 'Low' or shap_series.get('Motivation_Level', 0) < -0.1) else 0
    needs_study      = 1 if (student_raw.get('Hours_Studied', 20) < 15 or shap_series.get('Hours_Studied', 0) < -0.5) else 0
    needs_sleep      = 1 if (student_raw.get('Sleep_Hours', 7) < 6 or shap_series.get('Sleep_Hours', 0) < -0.2) else 0
    needs_resources  = 1 if (student_raw.get('Access_to_Resources') == 'Low' or shap_series.get('Access_to_Resources', 0) < -0.3) else 0

    return np.array([needs_motivation, needs_study, needs_sleep, needs_resources], dtype=float)

In [27]:
resource_vectors = np.array(
    [[r['motivation'], r['hours_studied'], r['sleep'], r['resources_access']]
     for r in resources],
    dtype=float
)

In [29]:
risk_results = []

# نفترض أننا نختبر أول 50 طالب في X_test
for i in range(len(X_test[:50])):
    student_raw = X_test.iloc[i].to_dict()
    student_scaled = X_test_scaled[i:i+1]
    
    # 1. توقع الاحتمالية
    prob_fail = model.predict_proba(student_scaled)[0][0] # Class 0: Fail
    
    # 2. جلب قيم SHAP لهذا الطالب
    student_shap = explainer.shap_values(student_scaled)[0]
    
    # 3. بناء الـ Vector والحصول على التوصيات
    # استخراج أسماء الأعمدة من الباكيج اللي حملناه

    student_vec = build_smart_student_vector(student_raw, student_shap, feature_columns).reshape(1, -1)
    
    # حساب التشابه (Cosine Similarity)
    similarities = cosine_similarity(student_vec, resource_vectors)[0]
    top_res_idx = np.argsort(similarities)[::-1][:2] # أحسن موردين
    
    # تخزين النتيجة
    risk_results.append({
        'student_id': i,
        'fail_prob': prob_fail,
        'risk_level': 'HIGH' if prob_fail > 0.6 else ('MEDIUM' if prob_fail > 0.4 else 'LOW'),
        'top_recommendation': resources[top_res_idx[0]]['title'] if similarities[top_res_idx[0]] > 0 else "General Coaching"
    })

# تحويلها لـ DataFrame وعرضها
risk_df = pd.DataFrame(risk_results)
print(risk_df.head(10))

   student_id     fail_prob risk_level  \
0           0  9.617880e-01       HIGH   
1           1  1.852025e-01        LOW   
2           2  1.474862e-09        LOW   
3           3  1.487033e-05        LOW   
4           4  5.250485e-04        LOW   
5           5  2.774783e-07        LOW   
6           6  2.054211e-02        LOW   
7           7  5.923340e-01     MEDIUM   
8           8  1.327112e-01        LOW   
9           9  3.764917e-06        LOW   

                                  top_recommendation  
0                                   General Coaching  
1                                   General Coaching  
2                                   General Coaching  
3                                   General Coaching  
4             Calm — Sleep & Relaxation for Students  
5                                   General Coaching  
6             Calm — Sleep & Relaxation for Students  
7       YouTube EDU — Free Educational Video Library  
8  Brilliant.org — Problem-Solving & Criti

In [41]:
risk_results = []

for i in range(len(X_test[:50])):
    student_raw = X_test.iloc[i].to_dict()
    student_scaled = X_test_scaled[i:i+1]
    
    # 1. حساب الاحتمالية والمستوى
    # نأخذ الاحتمالية ونحولها لنسبة مئوية للعرض
    prob_fail = float(model.predict_proba(student_scaled)[0][0]) 
    
    if prob_fail >= 0.7:
        risk_level = 'HIGH'
    elif prob_fail >= 0.4:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'
    
    # 2. تحليل SHAP
    student_shap = explainer.shap_values(student_scaled)[0]
    
    # 3. محرك التوصيات (Smart Vector)
    student_vec = build_smart_student_vector(student_raw, student_shap, feature_columns).reshape(1, -1)
    similarities = cosine_similarity(student_vec, resource_vectors)[0]
    
    # جلب أفضل 3 توصيات
    top_3_indices = np.argsort(similarities)[::-1][:3]
    top_3_names = [resources[idx]['title'] for idx in top_3_indices]

    # 4. تجميع كل البيانات في السجل
    risk_results.append({
        'student_id': i,
        'fail_probability': round(prob_fail * 100, 2), # عرضها كنسبة مئوية 96.17
        'risk_level': risk_level,
        'recommendation_1': top_3_names[0],
        'recommendation_2': top_3_names[1],
        'recommendation_3': top_3_names[2]
    })

# تحويل لـ DataFrame
final_risk_df = pd.DataFrame(risk_results)

# حفظ الملف للمرحلة الرابعة
final_risk_df.to_csv('asse_artifacts/final_student_report.csv', index=False)

print("✅ التقارير جاهزة!")
print(final_risk_df.head())

✅ التقارير جاهزة!
   student_id  fail_probability risk_level  \
0           0             96.18       HIGH   
1           1             18.52        LOW   
2           2              0.00        LOW   
3           3              0.00        LOW   
4           4              0.05        LOW   

                          recommendation_1  \
0  Notion — Student Study Planner Template   
1  Notion — Student Study Planner Template   
2  Notion — Student Study Planner Template   
3  Notion — Student Study Planner Template   
4   Calm — Sleep & Relaxation for Students   

                                  recommendation_2  \
0   Duolingo — Language Learning (Cognitive Boost)   
1   Duolingo — Language Learning (Cognitive Boost)   
2   Duolingo — Language Learning (Cognitive Boost)   
3   Duolingo — Language Learning (Cognitive Boost)   
4  Headspace — Student Mindfulness & Stress Relief   

                                    recommendation_3  
0                   Coursera — Learning How to L

In [43]:
# دمج بيانات الطلاب الأصلية مع نتايج التوقعات
final_report_with_data = pd.concat([X_test.reset_index(drop=True), final_risk_df], axis=1)
final_report_with_data.to_csv('asse_artifacts/full_instructor_report.csv', index=False)